In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip -q fa25_ece408_ClosedAI-salient_awq.zip

In [3]:
%cd fa25_ece408_ClosedAI-salient_awq

/content/fa25_ece408_ClosedAI-salient_awq


In [15]:
!make clean && make next_token_generation

rm -f *.o *.err *.out output_verification_rand test_gpt2 model_output next_token_generation local_attn_verify benchmark_attention
nvcc -O3 -arch=sm_80 -std=c++11 -rdc=true -g -lineinfo -I/usr/local/cuda/include -c next_token_generation.cu -o next_token_generation.o
nvcc -O3 -arch=sm_80 -std=c++11 -rdc=true -g -lineinfo -o next_token_generation next_token_generation.o -L/usr/local/cuda/lib64 -lcublas -lcublasLt  -lnvToolsExt


In [11]:
import subprocess

def run_command(cmd, verbose=True):
    """Execute a shell command and return success status"""
    if verbose:
        print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True)
    return result.returncode == 0

In [5]:
import tiktoken

# ----------------------------
# CONFIG
# ----------------------------
RECORD_DELIM = "\x1e"
MAX_TOKENS = 256
N_SAMPLES = 128
OUTPUT_FILE = "input.txt"

# Use the GPT-2 BPE tokenizer
enc = tiktoken.get_encoding("gpt2")

# ----------------------------
# SAFE CHUNKING FUNCTION
# ----------------------------
def chunk_text(text, max_tokens=MAX_TOKENS):
    """
    Split text into chunks of <= max_tokens without splitting tokens mid-way.
    Returns a list of token lists.
    """
    tokens = enc.encode(text, disallowed_special=())
    chunks = []

    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunks.append(tokens[start:end])
        start = end

    return chunks

# ----------------------------
# MAIN DATASET GENERATOR
# ----------------------------
def generate_calib_dataset(lines, n_samples=N_SAMPLES, max_tokens=MAX_TOKENS, output_file=OUTPUT_FILE):
    all_samples = []

    with open(output_file, "w", encoding="utf-8") as f:
        n_written = 0

        for line in lines:
            line = line.strip()
            if not line:
                continue

            # Split the line safely into chunks
            token_chunks = chunk_text(line, max_tokens=max_tokens)

            for chunk_tokens in token_chunks:
                # Decode back to string (C tokenizer expects text)
                chunk_text_str = enc.decode(chunk_tokens)

                # Remove record delimiter if accidentally present
                safe_text = chunk_text_str.replace(RECORD_DELIM, "")

                # Write sample + delimiter
                f.write(safe_text + RECORD_DELIM)
                all_samples.append(safe_text)

                n_written += 1
                if n_written >= n_samples:
                    print(f"Written {n_written} samples to {output_file}")
                    return all_samples

    print(f"Written {n_written} samples to {output_file}")
    return all_samples


In [6]:
from datasets import load_dataset
import random

dataset = load_dataset("mit-han-lab/pile-val-backup", split="validation")

# Convert to list of text lines
lines = [d["text"] for d in dataset]

# Shuffle
random.seed(42)
random.shuffle(lines)

# Generate calibration dataset
samples = generate_calib_dataset(lines, n_samples=N_SAMPLES, max_tokens=MAX_TOKENS, output_file=OUTPUT_FILE)

# Optional: inspect first 3 samples
samples[:3]


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/167 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


val.jsonl.zst:   0%|          | 0.00/471M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/214670 [00:00<?, ? examples/s]

Written 128 samples to input.txt


["1. Field of the Invention\nThe subject invention relates to movable partitions, also known as operable walls and, more particularly, to an improved operable wall system providing improved response to lateral loads.\n2. Description of Related Art\nOperable walls, particularly those used to divide large spaces such as convention centers, ballrooms, and gymnasiums, may be subjected to lateral loading of the wall panels as a result of earthquake or pressure differential between the rooms divided by the operable walls. The wall panels in question can have dimensions of up to 5 feet wide and up to 40 feet high. Loads can reach on the order of 20 pounds per square foot (psf). Based on 5'.times.40' panel dimensions, and 20 psf, the load to be resisted at the top and at the bottom of the panel is 2,000 pounds.\nSuch significant lateral loading can occur, for example, between spaces divided by an operable wall in an exhibit area where one of the rooms is adjacent to an exterior wall containing

In [16]:
!python3 run_sanitizer.py all

Running MEMCHECK
Command: compute-sanitizer --tool memcheck --show-backtrace yes --print-limit 100 --error-exitcode 1 --destroy-on-device-error kernel ./next_token_generation input.txt

========= COMPUTE-SANITIZER
GPT2 - Calibration mode ENABLED
+-----------------------+
| GPT2 MODEL PARAMETERS |
+-----------------------+----------------------------------------------------+
| max_sequence_length T | 1024                                               |
| vocab_size V          | 50257                                              |
| padded_vocab_size Vp  | 50304                                              |
| num_layers L          | 12                                                 |
| num_heads NH          | 12                                                 |
| channels C            | 768                                                |
| num_parameters        | 124475904                                          |
+-----------------------+---------------------------------------------

In [8]:
#!/usr/bin/env python3
"""
Script to read awq_channel_avgs.bin and compute top 1% salient channel indices.
Prints verification information about the data.
"""

import struct
import numpy as np
from pathlib import Path

def read_channel_averages(filename):
    """Read the binary file containing channel averages."""
    with open(filename, 'rb') as f:
        # Read header
        L = struct.unpack('i', f.read(4))[0]
        C = struct.unpack('i', f.read(4))[0]
        total_tokens = struct.unpack('Q', f.read(8))[0]  # size_t = unsigned long long (8 bytes)

        # Read averages
        total_channels = L * C * 7  # L*C + L*C + L*C + L*4*C
        averages = np.frombuffer(f.read(total_channels * 4), dtype=np.float32)

        return L, C, total_tokens, averages

def compute_salient_indices(averages, L, C):
    """Compute top 1% salient channel indices per layer per tensor type."""
    k_c = int(np.ceil(0.01 * C))      # top 1% of C
    k_4c = int(np.ceil(0.01 * 4 * C)) # top 1% of 4*C

    # Layout: [L*C ln1] [L*C atty] [L*C ln2] [L*4*C fch_gelu]
    offset_ln1 = 0
    offset_atty = L * C
    offset_ln2 = 2 * L * C
    offset_fch_gelu = 3 * L * C

    # Allocate arrays for indices
    salient_ln1 = np.zeros((L, k_c), dtype=np.uint32)
    salient_atty = np.zeros((L, k_c), dtype=np.uint32)
    salient_ln2 = np.zeros((L, k_c), dtype=np.uint32)
    salient_fch_gelu = np.zeros((L, k_4c), dtype=np.uint32)

    for l in range(L):
        # ln1: top k_c channels from layer l
        segment = averages[offset_ln1 + l*C : offset_ln1 + (l+1)*C]
        salient_ln1[l] = np.argpartition(segment, -k_c)[-k_c:]
        salient_ln1[l] = salient_ln1[l][np.argsort(segment[salient_ln1[l]])[::-1]]

        # atty: top k_c channels from layer l
        segment = averages[offset_atty + l*C : offset_atty + (l+1)*C]
        salient_atty[l] = np.argpartition(segment, -k_c)[-k_c:]
        salient_atty[l] = salient_atty[l][np.argsort(segment[salient_atty[l]])[::-1]]

        # ln2: top k_c channels from layer l
        segment = averages[offset_ln2 + l*C : offset_ln2 + (l+1)*C]
        salient_ln2[l] = np.argpartition(segment, -k_c)[-k_c:]
        salient_ln2[l] = salient_ln2[l][np.argsort(segment[salient_ln2[l]])[::-1]]

        # fch_gelu: top k_4c channels from layer l
        segment = averages[offset_fch_gelu + l*4*C : offset_fch_gelu + (l+1)*4*C]
        salient_fch_gelu[l] = np.argpartition(segment, -k_4c)[-k_4c:]
        salient_fch_gelu[l] = salient_fch_gelu[l][np.argsort(segment[salient_fch_gelu[l]])[::-1]]

    return salient_ln1, salient_atty, salient_ln2, salient_fch_gelu, k_c, k_4c

def main():
    # Read the binary file
    filename = 'awq_channel_avgs.bin'

    if not Path(filename).exists():
        print(f"Error: {filename} not found!")
        return

    print(f"Reading {filename}...")
    L, C, total_tokens, averages = read_channel_averages(filename)

    print("\n" + "="*70)
    print("FILE HEADER INFORMATION")
    print("="*70)
    print(f"Number of layers (L):     {L}")
    print(f"Channels per layer (C):   {C}")
    print(f"Total tokens processed:   {total_tokens}")
    print(f"Total channels stored:    {len(averages)} (expected: {L*C*7})")

    # Statistics
    print("\n" + "="*70)
    print("AVERAGE MAGNITUDE STATISTICS")
    print("="*70)
    print(f"Min average magnitude:    {averages.min():.6f}")
    print(f"Max average magnitude:    {averages.max():.6f}")
    print(f"Mean average magnitude:   {averages.mean():.6f}")
    print(f"Std average magnitude:    {averages.std():.6f}")

    # Compute salient indices
    print("\n" + "="*70)
    print("COMPUTING SALIENT INDICES (Top 1%)")
    print("="*70)
    salient_ln1, salient_atty, salient_ln2, salient_fch_gelu, k_c, k_4c = \
        compute_salient_indices(averages, L, C)

    print(f"k_c (top 1% of {C}):       {k_c}")
    print(f"k_4c (top 1% of {4*C}):    {k_4c}")
    print(f"Total indices per layer:   {3*k_c + k_4c}")
    print(f"Total indices all layers:  {L * (3*k_c + k_4c)}")

    # Print sample salient indices for layer 0
    print("\n" + "="*70)
    print("SAMPLE: Layer 0 Salient Indices")
    print("="*70)

    offset_ln1 = 0
    offset_atty = L * C
    offset_ln2 = 2 * L * C
    offset_fch_gelu = 3 * L * C

    print(f"\nln1 (first 5 salient channels):")
    for i in range(min(5, k_c)):
        idx = salient_ln1[0, i]
        mag = averages[offset_ln1 + idx]
        print(f"  Channel {idx:4d}: magnitude = {mag:.6f}")

    print(f"\natty (first 5 salient channels):")
    for i in range(min(5, k_c)):
        idx = salient_atty[0, i]
        mag = averages[offset_atty + idx]
        print(f"  Channel {idx:4d}: magnitude = {mag:.6f}")

    print(f"\nln2 (first 5 salient channels):")
    for i in range(min(5, k_c)):
        idx = salient_ln2[0, i]
        mag = averages[offset_ln2 + idx]
        print(f"  Channel {idx:4d}: magnitude = {mag:.6f}")

    print(f"\nfch_gelu (first 5 salient channels):")
    for i in range(min(5, k_4c)):
        idx = salient_fch_gelu[0, i]
        mag = averages[offset_fch_gelu + idx]
        print(f"  Channel {idx:4d}: magnitude = {mag:.6f}")

    # Print sample for last layer
    print("\n" + "="*70)
    print(f"SAMPLE: Layer {L-1} Salient Indices")
    print("="*70)

    print(f"\nln1 (first 5 salient channels):")
    for i in range(min(5, k_c)):
        idx = salient_ln1[L-1, i]
        mag = averages[offset_ln1 + (L-1)*C + idx]
        print(f"  Channel {idx:4d}: magnitude = {mag:.6f}")

    print(f"\nfch_gelu (first 5 salient channels):")
    for i in range(min(5, k_4c)):
        idx = salient_fch_gelu[L-1, i]
        mag = averages[offset_fch_gelu + (L-1)*4*C + idx]
        print(f"  Channel {idx:4d}: magnitude = {mag:.6f}")

    print("\n" + "="*70)
    print("VERIFICATION COMPLETE")
    print("="*70)

if __name__ == "__main__":
    main()


Reading awq_channel_avgs.bin...

FILE HEADER INFORMATION
Number of layers (L):     12
Channels per layer (C):   768
Total tokens processed:   1758684
Total channels stored:    64512 (expected: 64512)

AVERAGE MAGNITUDE STATISTICS
Min average magnitude:    0.000000
Max average magnitude:    0.000000
Mean average magnitude:   0.000000
Std average magnitude:    0.000000

COMPUTING SALIENT INDICES (Top 1%)
k_c (top 1% of 768):       8
k_4c (top 1% of 3072):    31
Total indices per layer:   55
Total indices all layers:  660

SAMPLE: Layer 0 Salient Indices

ln1 (first 5 salient channels):
  Channel   39: magnitude = 0.000000
  Channel   38: magnitude = 0.000000
  Channel   37: magnitude = 0.000000
  Channel   36: magnitude = 0.000000
  Channel   35: magnitude = 0.000000

atty (first 5 salient channels):
  Channel   39: magnitude = 0.000000
  Channel   38: magnitude = 0.000000
  Channel   37: magnitude = 0.000000
  Channel   36: magnitude = 0.000000
  Channel   35: magnitude = 0.000000

ln2 